# SAE Autoresearch Experiment Analysis

Analysis of autonomous SAE experimentation results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 7 columns)
df = pd.read_csv("results.tsv", sep="\t")
for col in ["sae_score", "val_mse", "val_l0", "dead_frac"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
# Experiment outcomes
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments with full metrics
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    print(f"  #{i:3d}  score={row['sae_score']:.4f}  mse={row['val_mse']:.6f}  "
          f"l0={row['val_l0']:.4f}  dead={row['dead_frac']:.4f}  {row['description']}")

In [ ]:
# SAE Score Over Time
fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)
baseline_score = valid.loc[0, "sae_score"]

# Discarded as faint background
disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["sae_score"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Kept experiments as prominent green dots
kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["sae_score"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running maximum step line (higher is better)
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_scores = valid.loc[kept_mask, "sae_score"]
running_max = kept_scores.cummax()
ax.step(kept_idx, running_max, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment
for idx, score in zip(kept_idx, kept_scores):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (idx, score),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("SAE Score (higher is better)", fontsize=12)
ax.set_title(f"SAE Autoresearch Progress: {len(df)} Experiments, {len(kept_v)} Kept", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

In [ ]:
# MSE vs L0 Pareto Plot
fig, ax = plt.subplots(figsize=(10, 8))

valid = df[df["status"] != "CRASH"].copy()

colors = {"KEEP": "#2ecc71", "DISCARD": "#cccccc"}
sizes = {"KEEP": 50, "DISCARD": 15}

for status in ["DISCARD", "KEEP"]:
    subset = valid[valid["status"] == status]
    ax.scatter(subset["val_mse"], subset["val_l0"],
              c=colors[status], s=sizes[status],
              label=status.capitalize(), alpha=0.7,
              edgecolors="black" if status == "KEEP" else "none",
              linewidths=0.5, zorder=3 if status == "KEEP" else 2)

ax.set_xlabel("Validation MSE (lower is better)", fontsize=12)
ax.set_ylabel("Validation L0 (lower is better)", fontsize=12)
ax.set_title("MSE vs L0 Tradeoff (Pareto Frontier)", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# Dead Features Over Time (kept experiments only)
kept = df[df["status"] == "KEEP"].copy().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(len(kept)), kept["dead_frac"], "o-", color="#e74c3c", markersize=6)
for i, row in kept.iterrows():
    ax.annotate(f"{row['dead_frac']:.3f}", (i, row["dead_frac"]),
                textcoords="offset points", xytext=(0, 8), fontsize=8, ha="center")

ax.set_xlabel("Kept Experiment #", fontsize=12)
ax.set_ylabel("Dead Feature Fraction", fontsize=12)
ax.set_title("Dead Features Across Kept Experiments", fontsize=14)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Summary Statistics
kept = df[df["status"] == "KEEP"].copy()
baseline = df.iloc[0]["sae_score"]
best = kept["sae_score"].max()
best_row = kept.loc[kept["sae_score"].idxmax()]

print(f"Baseline sae_score:  {baseline:.6f}")
print(f"Best sae_score:      {best:.6f}")
print(f"Total improvement:   {best - baseline:+.6f} ({(best - baseline) / abs(baseline) * 100:+.2f}%)")
print(f"Best experiment:     {best_row['description']}")
print()
print(f"Best MSE:  {best_row['val_mse']:.6f}")
print(f"Best L0:   {best_row['val_l0']:.6f}")
print(f"Best dead: {best_row['dead_frac']:.6f}")
print()

print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: score={row['sae_score']:.4f}  {desc}")

In [ ]:
# Top Hits (Kept Experiments by Improvement)
kept = df[df["status"] == "KEEP"].copy()
kept["prev_score"] = kept["sae_score"].shift(1)
kept["delta"] = kept["sae_score"] - kept["prev_score"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'Score':>10}  {'MSE':>10}  {'L0':>8}  Description")
print("-" * 90)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['sae_score']:.6f}  "
          f"{row['val_mse']:.6f}  {row['val_l0']:.4f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  {'':>10}  {'':>8}  TOTAL improvement over baseline")